# 13 Tuning Bark Detector

The model tuned from methods/settings in events.py:
- threshold: how high the score must be to count as a dog sound
- median_frames: smooth the scores so one odd 40 ms blip is ignored
- max_gap_s: if two sounds are this close, join them into one
- min_event_s: throw away sounds shorter than this

In [1]:
import contextlib
import io
import itertools
import warnings

import numpy as np
import pandas as pd

from woofdata.loader import load_sample
from woofdata.path import data_path, load_data_dir_from_env_file
from woofdata.tasks import task_rows
from woofdecoder.branches.bark_detector.events import (
    DetectionSettings,
    events_to_frames,
    find_dog_events,
)
from woofdecoder.branches.bark_detector.model import load_bark_detector
from woofdecoder.branches.bark_detector.predict import detect_sed_probs
from wooftrainer.eval.metrics import event_scores, frame_scores, matched_event_scores

load_data_dir_from_env_file()
TASK = "bark_detector"
CACHE_DIR = data_path("cache/bark_detector")

## Load the Model's Scores

Tuning only changes the clean-up steps, so the model only has to run **once** per dataset. The scores are saved to a file, and every later run just loads that file. Notebook 12 already saved playpant val.

Three datasets are used for choosing (all "val", the practice set):

- **playpant**: recordings of playing dogs. Almost all of it is panting.
- **AudioSet**: mostly barks, growls and whines, plus 300 clips with no dog at all.
- **ESC-50**: everyday sounds (rain, doors, babies...). Used to check for false alarms.

In [2]:
def load_scores(source, split):
    path = CACHE_DIR / f"{source}_{split}_zero_shot.npz"

    if not path.is_file():
        # Run the model once and save its scores.
        # loading prints the whole network --> hide
        with contextlib.redirect_stdout(io.StringIO()), warnings.catch_warnings():
            warnings.simplefilter("ignore")
            detector = load_bark_detector(verbose=False)
            rows = task_rows(TASK, split, sources=[source])
            samples = [load_sample(TASK, row) for _, row in rows.iterrows()]
            probs = [detect_sed_probs(detector, s.audio)[: len(s.targets)] for s in samples]

        np.savez(path, probs=np.stack(probs), targets=np.stack([s.targets for s in samples]),
                 is_dog=np.array([s.is_dog for s in samples]), clip_ids=rows["clip_id"].to_numpy())
    
    saved = np.load(path, allow_pickle=True)

    dog_score = saved["probs"].max(axis=2)
    truth = saved["targets"].any(axis=2)

    return dog_score, truth, saved

playpant = load_scores("playpant", "val")[:2]
audioset = load_scores("audioset_strong_eval", "val")[:2]
esc50_score, _, esc50_saved = load_scores("esc50", "val")
esc50_is_dog = esc50_saved["is_dog"]

print(f"playpant: {len(playpant[0])} clips, AudioSet: {len(audioset[0])} clips, ESC-50: {len(esc50_score)} clips")

playpant: 403 clips, AudioSet: 627 clips, ESC-50: 403 clips


## Enhanced Count Events

Event score from notebook 12 has error --> join everything into one long event to find more accurate sound events

Example below shows 2 barks (before and after fix F1 scores)

In [3]:
real   = np.array([[1, 1, 0, 0, 1, 1, 0, 0]], dtype=bool) # two barks
guess  = np.array([[1, 1, 1, 1, 1, 1, 0, 0]], dtype=bool) # one long event

print("old event F1:", event_scores(real, guess)["f1"])
print("new event F1:", round(matched_event_scores(real, guess)["f1"], 2))

old event F1: 1.0
new event F1: 0.67


The old score gives a perfect 1.0, even though we missed that there were two barks.

The new score (from matched_event_scores) lets each guess match only one real sound, so it gives 0.67.

In [4]:
def score(settings, dog_score, truth):
    guess_rows = []
    for row in dog_score:
        events = find_dog_events(row, settings)
        frames = events_to_frames(events, len(row))
        guess_rows.append(frames)

    guess = np.stack(guess_rows)
    
    return frame_scores(truth, guess)["f1"], matched_event_scores(truth, guess)["f1"]

## Try Every Combination

- threshold: 0.02 to 0.2
- median_frames: 1 (off) to 9 (must be odd so there is a middle frame)
- max_gap_s: 0 to 1 s --> closes gap between (then joins)
- min_event_s: 0 to 0.5 --> most real dog sounds last about 1 s

In [5]:
rows = []
for values in itertools.product(
    [0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2], # threshold
    [1, 3, 5, 9], # median_frames
    [0.0, 0.08, 0.16, 0.3, 0.5, 1.0], # max_gap_s
    [0.0, 0.08, 0.16, 0.3, 0.5] # min_event_s
):
    settings = DetectionSettings(*values)

    playpant_frame, playpant_event = score(settings, *playpant)
    audioset_frame, audioset_event = score(settings, *audioset)

    rows.append({
        "settings": settings,
        "playpant_frame": playpant_frame,
        "playpant_event": playpant_event,
        "audioset_frame": audioset_frame,
        "audioset_event": audioset_event
    })

grid = pd.DataFrame(rows)
grid["frame_avg"] = (grid["playpant_frame"] + grid["audioset_frame"]) / 2
grid["event_avg"] = (grid["playpant_event"] + grid["audioset_event"]) / 2
print(f"{len(grid)} combinations tried")

840 combinations tried


**How we picked:**

Take the best event F1, but only among settings whose frame F1 is within 0.01 of the best frame F1.

- Event F1 matters most, because the next parts of the app get one clip per event.
- The frame F1 rule stops a setting from winning by cutting sounds too short or making them too long.

In [6]:
def pick(frame_column, event_column):
    good_enough = grid[grid[frame_column] >= grid[frame_column].max() - 0.01]
    return good_enough.loc[good_enough[event_column].idxmax(), "settings"]

playpant_only = pick("playpant_frame", "playpant_event")
chosen = pick("frame_avg", "event_avg")

print("playpant only:", playpant_only)
print("chosen: ", chosen)

playpant only: DetectionSettings(threshold=0.08, median_frames=1, max_gap_s=0.3, min_event_s=0.0)
chosen:  DetectionSettings(threshold=0.08, median_frames=1, max_gap_s=0.0, min_event_s=0.16)


## Compare

- default: threshold 0.5, the code's current setting
- nb12: threshold 0.1, the best from notebook 12
- playpant_only and chosen: from above

The ESC-50 row shows how often model fires false alarm.

In [7]:
candidates = {
    "default": DetectionSettings(),
    "nb12": DetectionSettings(threshold=0.1),
    "playpant_only": playpant_only,
    "chosen": chosen,
}

def false_alarm_rate(settings):
    fires = np.array([len(find_dog_events(row, settings)) > 0 for row in esc50_score])
    return fires[~esc50_is_dog].mean()

table = {}
for name, settings in candidates.items():
    table[name] = {
        "playpant event F1": score(settings, *playpant)[1],
        "AudioSet event F1": score(settings, *audioset)[1],
        "ESC-50 false alarms": false_alarm_rate(settings),
    }
pd.DataFrame(table).round(2)

,default,nb12,playpant_only,chosen
playpant event F1,0.10,0.40,0.49,0.46
AudioSet event F1,0.41,0.62,0.55,0.63
ESC-50 false alarms,0.00,0.00,0.00,0.00


- Tuning on playpant alone helps pants but hurts barks (AudioSet drops from 0.62 to 0.55).
- Pants are long and steady, so playpant likes joining sounds together.
- Barks come in quick runs ("woof woof woof"), and joining them turns 3 barks into 1.
- So chosen gives playpant and AudioSet equal weight --> best of both
- No false alarms on ESC-50, even with the low threshold.

## Final Check on Test

The test set was never used for choosing. It shows how the detector does on clips it has not seen.

In [8]:
test = load_scores("playpant", "test")[:2]

pd.DataFrame(
    {name: dict(zip(["frame F1", "event F1"], score(settings, *test))) for name, settings in candidates.items()}
).round(2)

,default,nb12,playpant_only,chosen
frame F1,0.05,0.72,0.76,0.75
event F1,0.16,0.46,0.53,0.52


chosen beats nb12 on test too (frame 0.75 vs 0.72, event 0.52 vs 0.46)
- playpant_only is a tiny bit higher here because the test set is mostly pants